# ISLES'24 download (Phase 0)

Downloads `train.7z` from Zenodo record 17652035 into `StrokeCopilot/raw/` on Drive and verifies its md5. Run once. This is a data-prep exception to the one rule in plan §4.3: it writes the archive through the Drive mount, because nothing trains on it here.

**Before you start**
- Use a CPU runtime: no GPU is needed, and this leaves your GPU allowance alone.
- `StrokeCopilot/raw/` must already exist in your Drive, or as a shortcut under `MyDrive`. A shared folder counts against its owner's storage.
- The archive is 99 GB. Keep the tab open and the computer awake.

**If it stops:** reconnect and use Runtime → Run all. The download continues from what is already on Drive, and finished steps are skipped.

The notebook is self-contained, so it works before the repo is pushed. Nothing here needs the `strokecopilot` package.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings

In [ ]:
import hashlib, json, os, shutil, subprocess, time, urllib.request
from datetime import datetime, timezone

RECORD_ID = 17652035
ARCHIVE_KEY = 'train.7z'
ARCHIVE_SIZE = 99022114670
ARCHIVE_MD5 = '4959a5dd2438d53e3c86d6858484e781'
API_URL = 'https://zenodo.org/api/records/17652035'
FILE_URL = 'https://zenodo.org/api/records/17652035/files/train.7z/content'
RAW_DIR = '/content/drive/MyDrive/StrokeCopilot/raw'
POLL_SECONDS = 60
RETRY_SECONDS = 30
MAX_ATTEMPTS = 50

PART = os.path.join(RAW_DIR, 'isles24_train.7z.part')
FINAL = os.path.join(RAW_DIR, 'isles24_train.7z')
RECORD = os.path.join(RAW_DIR, 'isles24_download.json')

def size_of(path):
    return os.path.getsize(path) if os.path.exists(path) else 0

## 3. Pre-flight checks
Stops if the folder is missing, Drive is too full, or Zenodo no longer matches the pinned file.

In [ ]:
if not os.path.isdir(RAW_DIR):
    raise FileNotFoundError(f'{RAW_DIR} not found. Create the Drive folder or add its shortcut first (plan §4.2).')

with urllib.request.urlopen(API_URL, timeout=60) as r:
    files = {f['key']: f for f in json.load(r)['files']}
remote = files[ARCHIVE_KEY]
if remote['size'] != ARCHIVE_SIZE or remote['checksum'] != f'md5:{ARCHIVE_MD5}':
    raise ValueError(f'Zenodo no longer matches the pinned file: {remote}')

needed = 0 if os.path.exists(FINAL) else ARCHIVE_SIZE - size_of(PART)
free = shutil.disk_usage(RAW_DIR).free
print(f'Still to download: {needed / 1e9:.1f} GB. Free space reported: {free / 1e9:.1f} GB.')
if needed > free:
    raise OSError('Not enough Drive space for the archive.')

## 4. Download
Prints progress every minute. Saves to a `.part` file, so an unverified archive never looks finished.

In [ ]:
def download():
    if os.path.exists(FINAL):
        return print('Archive already verified and in place. Skipping download.')
    for attempt in range(1, MAX_ATTEMPTS + 1):
        have = size_of(PART)
        if have == ARCHIVE_SIZE:
            return print('Download complete.')
        if have > ARCHIVE_SIZE:
            raise ValueError(f'{PART} is larger than expected. Check it, then delete it by hand.')
        cmd = ['curl', '-sS', '-L', '--fail', '-C', '-', '--speed-limit', '1024', '--speed-time', '300', '-o', PART, FILE_URL]
        proc = subprocess.Popen(cmd)
        while proc.poll() is None:
            time.sleep(POLL_SECONDS)
            print(f'{size_of(PART) / 1e9:.1f} / {ARCHIVE_SIZE / 1e9:.1f} GB')
        if size_of(PART) != ARCHIVE_SIZE:
            print(f'Attempt {attempt} stopped (curl exit {proc.returncode}). Resuming shortly.')
            time.sleep(RETRY_SECONDS)
    if size_of(PART) != ARCHIVE_SIZE:
        raise RuntimeError('Download did not finish. Run this notebook again to resume.')

download()

## 5. Verify and record
Uploads anything still pending, computes the md5 from Drive, then renames the file to `isles24_train.7z` and writes `isles24_download.json`. Uploading and hashing can take a while. Drive may ask you to re-authorise.

In [ ]:
def file_md5(path):
    digest, done = hashlib.md5(), 0
    with open(path, 'rb') as f:
        while chunk := f.read(64 * 1024 * 1024):
            digest.update(chunk)
            done += len(chunk)
            if done % (10 * 1024 ** 3) < len(chunk):
                print(f'Hashed {done / 1e9:.0f} GB')
    return digest.hexdigest()

def verify():
    if os.path.exists(RECORD):
        return print('Already verified:', open(RECORD).read())
    drive.flush_and_unmount()  # push everything to Drive, then hash what is stored there
    drive.mount('/content/drive', force_remount=True)
    source = FINAL if os.path.exists(FINAL) else PART
    actual = file_md5(source)
    if actual != ARCHIVE_MD5:
        raise ValueError(f'md5 mismatch: got {actual}, expected {ARCHIVE_MD5}. {source} was kept. Check it, then delete it by hand and run again.')
    if source == PART:
        os.replace(PART, FINAL)
    record = {'record_id': RECORD_ID, 'doi': f'10.5281/zenodo.{RECORD_ID}', 'file': ARCHIVE_KEY, 'size_bytes': ARCHIVE_SIZE, 'md5': actual, 'verified_utc': datetime.now(timezone.utc).isoformat()}
    with open(RECORD, 'w') as f:
        json.dump(record, f, indent=2)
    drive.flush_and_unmount()
    print('Verified and saved. Copy this into docs/DATA.md:', json.dumps(record, indent=2))

verify()